<a href="https://colab.research.google.com/github/Amankumar1456/Dofus_Pulpit/blob/main/embeddings_hands_on.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Embeddings: hands-on lab

One practical notebook for the whole session. Upload this file to Google Colab, then run the sections in order.

You will: inspect embedding vectors, rank documents with cosine similarity, chunk a long page, and build a small embedding index.

## 1. Vector basics: dimension and normalization

In [ ]:
import numpy as np

embedding = np.array([0.18, -0.42, 0.31, 0.09])
normalized = embedding / np.linalg.norm(embedding)

print('dimension:', embedding.size)
print('raw magnitude:', round(np.linalg.norm(embedding), 3))
print('normalized magnitude:', round(np.linalg.norm(normalized), 3))
print('normalized vector:', normalized.round(3))

dimension: 4
raw magnitude: 0.559
normalized magnitude: 1.0
normalized vector: [ 0.322 -0.751  0.554  0.161]


**Try it:** change the values or add dimensions. Dimension is the number of values stored for each text. Normalization makes vector length equal to one, so dot product becomes cosine similarity.

## 2. Cosine similarity: rank a toy corpus

In [ ]:
documents = {
    'Change your account password': np.array([0.95, 0.15]),
    'Recover a locked account': np.array([0.82, 0.33]),
    'Update a billing address': np.array([0.12, 0.94]),
    'Download an invoice': np.array([0.18, 0.78]),
}
query = np.array([0.91, 0.24])  # Reset my password

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

ranking = sorted(((name, cosine(query, vector)) for name, vector in documents.items()),
                 key=lambda item: item[1], reverse=True)
for rank, (name, score) in enumerate(ranking, start=1):
    print(f'{rank}. {name:<30} cosine={score:.3f}')

1. Change your account password   cosine=0.995
2. Recover a locked account       cosine=0.992
3. Download an invoice            cosine=0.466
4. Update a billing address       cosine=0.375


**Try it:** replace `query` with `np.array([0.2, 0.95])`. Which document becomes the nearest neighbour?

## 3. Chunking: respect the token budget

In [ ]:
# A stand-in for a 1,260-token help-center page.
page_tokens = [f'token_{i}' for i in range(1260)]
max_sequence_length = 512

print('tokens embedded as one page:', len(page_tokens[:max_sequence_length]))
print('tokens lost to truncation:', len(page_tokens) - max_sequence_length)

def chunk(items, size=320, overlap=48):
    step = size - overlap
    return [items[start:start + size] for start in range(0, len(items), step)]

chunks = chunk(page_tokens)
for number, piece in enumerate(chunks, start=1):
    print(f'chunk {number}: {piece[0]} → {piece[-1]} ({len(piece)} tokens)')

tokens embedded as one page: 512
tokens lost to truncation: 748
chunk 1: token_0 → token_319 (320 tokens)
chunk 2: token_272 → token_591 (320 tokens)
chunk 3: token_544 → token_863 (320 tokens)
chunk 4: token_816 → token_1135 (320 tokens)
chunk 5: token_1088 → token_1259 (172 tokens)


**Try it:** change chunk size and overlap. In production, first split on headings or paragraphs, then enforce the token limit. Store the heading and source URL with each chunk.

## 4. End to end: embed chunks and retrieve them

This first download in Colab may take a minute. The same model and normalization setting are used for documents and the query.

In [ ]:
!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

settings = {
    'model': 'sentence-transformers/all-MiniLM-L6-v2',
    'normalize_embeddings': True,
    'chunk_tokens': 320,
    'overlap': 48,
}
model = SentenceTransformer(settings['model'])

corpus = [
    {'heading': 'Rate limits', 'url': '/api/rate-limits', 'text': 'A 429 response means the request rate limit was exceeded. Wait for retry-after before sending another request.'},
    {'heading': 'Authentication', 'url': '/api/auth', 'text': 'Use an API key in the Authorization header. Rotate a leaked key immediately.'},
    {'heading': 'Billing', 'url': '/account/billing', 'text': 'Update the billing address from the account settings page.'},
]
texts = [f"{item['heading']}: {item['text']}" for item in corpus]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
document_vectors = model.encode(texts, normalize_embeddings=settings['normalize_embeddings'])
query = 'Why did I get a 429 and when can I retry?'
query_vector = model.encode(query, normalize_embeddings=settings['normalize_embeddings'])
scores = document_vectors @ query_vector  # normalized: dot product = cosine similarity

for index in np.argsort(scores)[::-1]:
    print(f"{scores[index]:.3f}  {corpus[index]['heading']:<16} {corpus[index]['url']}")

0.615  Rate limits      /api/rate-limits
0.113  Authentication   /api/auth
0.085  Billing          /account/billing


## 5. Production checklist

Persist the model name, normalization choice, chunking settings, and corpus version with your index. If you change the model or representation settings, re-embed the corpus before comparing it with new query vectors.